Ingolf Lepenies, lepy@tuta.io · November 2026 · sdata 1.6

# Das sdata-Format 2026

## Selbstbeschreibende Messdaten mit Einheiten, Semantik und Identität

**Zusammenfassung.** Messdaten verlieren auf dem Weg vom Prüfstand über
Tabellenkalkulation und Austauschdatei bis zur Auswertung regelmäßig ihren
Kontext: Einheiten stehen in einer Kopfzeile, die beim nächsten Export fehlt,
der Messbeginn steckt im Dateinamen, und welche Größe eine Spalte eigentlich
meint, weiß nur, wer die Messung gemacht hat. `sdata` ist eine quelloffene
Python-Bibliothek, die eine Tabelle zusammen mit ihren vollständig
beschriebenen Metadaten als ein Objekt führt und diese Einheit über alle
unterstützten Formate erhält. Dieser Beitrag beschreibt den Stand der Version
1.6, sechs Jahre nach der ersten Darstellung des Formats [1]. Teil I erläutert
Motivation, Designprinzipien und Datenmodell und ordnet `sdata` in bestehende
Standards ein. Teil II führt das Modell an einem durchgängigen Beispiel vor,
derselben Temperaturmessung wie 2020, und wird beim Erzeugen dieses Dokuments
gegen die installierte Version ausgeführt.

# Teil I — Motivation und Design

## 1 Ausgangslage

Eine Tabelle mit zwei Spalten, `time` und `temperature`, ist ohne Zusatzwissen
wertlos. Erst mit den Angaben „Zeit in Sekunden seit Messbeginn",
„Lufttemperatur in Grad Celsius, gemessen mit einem PT100" und „Messbeginn am
6. November 2026 um 9 Uhr" wird daraus eine Messung, die jemand anderes
nachvollziehen, weiterverarbeiten oder mit anderen Messungen vergleichen kann.

In der Praxis liegt dieses Zusatzwissen verstreut vor: in Kopfzeilen, in
Dateinamen, in einem Laborbuch oder gar nicht. Jede Konvertierung, etwa von
Excel nach CSV oder von CSV in eine Datenbank, ist eine Gelegenheit, es zu
verlieren. Die FAIR-Prinzipien [2] fordern deshalb, dass Daten auffindbar,
zugänglich, interoperabel und wiederverwendbar sind, und betonen dabei
ausdrücklich reichhaltige, maschinenlesbare Metadaten, die mit den Daten
verbunden bleiben.

`sdata` (*strukturierte Daten*) setzt genau dort an. Die Bibliothek verlangt
kein neues Dateiformat, sondern legt ein kleines Objektmodell über die
etablierten Formate (Parquet, CSV, HDF5, Frictionless Data Package, JSON-LD)
und sorgt dafür, dass die Metadaten in jedem davon mitreisen.

## 2 Was sich seit 2020 geändert hat

Die Darstellung von 2020 [1] beruhte auf `sdata` 0.8.4 und der Klasse
`sdata.Data`: einer Tabelle, Metadaten aus Schlüssel-Wert-Attributen mit
optionaler Einheit und einer Beschreibung. Dieses Grundmodell hat sich bewährt
und ist geblieben. Verändert hat sich fast alles darum herum:

* **Tabellenklasse.** `sdata.Data` ist abgekündigt und wird mit 2.0 entfernt.
  Ihre Rolle übernimmt `sdata.sclass.dataframe.DataFrame`, die zusätzlich
  Metadaten je Spalte führt.
* **Identität.** Statt einer frei vergebenen UUID erhält jedes Objekt eine
  deterministische, lesbare Kennung (SUUID), aus der eine
  Decentralized-Identifier-Form (`did:suuid:…`) abgeleitet wird.
* **Datentypen.** Attributwerte werden gegen eine zentrale Typtabelle geprüft
  und umgewandelt; neben Zahl und Text gibt es Zeitstempel, Datum, Dauer,
  exakte Dezimalzahl, Binärdaten, sprachmarkierte Texte und weitere.
* **Einheiten.** Ein eigenes Einheitenmodul kennt Dimensionen, rechnet um und
  bildet Einheiten auf das QUDT-Vokabular ab.
* **Semantik.** Metadaten lassen sich als JSON-LD mit etablierten Vokabularen
  (QUDT, BFO, PROV, DCAT, CSVW, schema.org) ausgeben und als W3C Verifiable
  Credential signieren.
* **Formate.** Hinzugekommen sind Parquet, Arrow/Feather, natives HDF5 und
  das Frictionless Data Package, jeweils mit eingebetteten Metadaten.
* **Arbeitsweise.** Größere Entscheidungen werden seit 2026 als RFC im
  Repository festgehalten (derzeit 19 Dokumente unter `docs/rfc/`); die
  Testabdeckung des Pakets liegt bei 100 %.

Das Paket ist weiterhin als *Alpha* klassifiziert. Version 1.6 enthielt
deshalb bewusst einige inkompatible Änderungen, die im Changelog einzeln
ausgewiesen sind. Beim Vorbereiten dieses Beitrags fielen zudem Fehler auf:
Metadaten mit Zeitstempeln, Datumswerten oder Binärdaten ließen sich nicht nach
JSON, Parquet, Data Package oder HDF5 schreiben (behoben in 1.6.1), und die
Ablage in einem Store oder einer SQL-Datenbank brach ab oder verfälschte solche
Werte (behoben in 1.6.2). Das Beispiel in Teil II nutzt genau diese Fälle.

## 3 Designprinzipien

**Die Metadaten reisen mit den Daten.** Jedes Ausgabeformat trägt die
vollständigen Metadaten in sich: Parquet in den Schema-Metadaten der Datei und
zusätzlich je Feld, HDF5 als Attribute der Gruppe und der einzelnen Datensätze,
das Data Package im Deskriptor. Wo ein Format keinen Platz dafür hat, wie bei
reinem CSV, schreibt `sdata` auf Wunsch eine Begleitdatei `<sname>.meta.jsonld`
daneben. Das Ziel sind verlustfreie Rundreisen: Was geschrieben wurde, lässt
sich mit derselben Bedeutung wieder einlesen.

**Identität ist deterministisch.** Die Kennung eines Objekts ergibt sich aus
seiner Klasse und seinem Namen über eine UUID der Version 5. Dieselbe Messung,
an zwei Orten unabhängig angelegt, erhält dieselbe Kennung; Groß- und
Kleinschreibung des Namens spielen dabei keine Rolle. Die Kennung ist zugleich
lesbar (`DataFrame__temperaturmessung_001__e9a7…`) und taugt als Dateiname.

**Typen sind explizit.** Jedes Attribut trägt einen Datentyp, und jeder
Datentyp hat eine feste Abbildung auf XML-Schema-Typen für JSON-LD. Die
Umwandlung ist standardmäßig nachsichtig (ein ungültiger Wert wird gemeldet
und nicht übernommen) und auf Wunsch streng (`strict=True` löst eine
Ausnahme aus).

**Einheiten sind Daten.** Eine Einheit ist kein Kommentar, sondern ein
auswertbares Feld. `sdata` kennt zu jeder unterstützten Einheit einen
Dimensionsvektor über sieben Achsen — Länge, Masse, Zeit, Temperatur, ebener
Winkel, Stromstärke und Stoffmenge — sowie Faktor und Versatz zur kohärenten
SI-Einheit. Damit lassen sich Einheiten prüfen, umrechnen und gegen ein
Schema validieren. Der ebene Winkel ist eine eigene Achse, damit `rad` und
`deg` ineinander umrechenbar, aber nicht mit dimensionslosen Größen
verwechselbar sind; die Lichtstärke ist (noch) nicht modelliert.

**Der Kern bleibt schlank.** Der Kern benötigt nur `numpy`, `pandas` und
`suuid`. Parquet (`pyarrow`), HDF5 (`h5py`), URI-Zugriff (`fsspec`),
erweiterte Einheiten (`pint`), RDF (`rdflib`) und JSON-Schema (`jsonschema`)
sind optionale Extras. Fehlt ein Extra, fällt `sdata` auf eine reine
Python-Lösung zurück, wo das möglich ist — JSON-LD ist bereits gültiges RDF,
und die Signatur von Credentials ist ohne Fremdbibliothek implementiert.

**Standards statt Eigenformat.** `sdata` erfindet weder ein Container- noch
ein Vokabularformat. Gespeichert wird in etablierten Formaten, beschrieben
mit etablierten Vokabularen; das Eigene ist nur die Zusammenführung.

**Fehler sind eindeutig.** Wer auf einen externen Inhalt zugreift, etwa eine
referenzierte Datei, erhält bei Problemen eine Ausnahme, die zwischen „gibt es
nicht", „darf ich nicht lesen" und „kann ich gerade nicht prüfen"
unterscheidet, statt eines stillen `False` oder `None`.

## 4 Datenmodell und Ablage

Alle `sdata`-Objekte bestehen aus drei Teilen: den **Metadaten**
(`metadata`), den eigentlichen **Daten** und einer **Beschreibung**
(`description`, Freitext oder Markdown). Die gemeinsame Basisklasse `Base`
stellt Identität, Metadaten und Serialisierung bereit; davon abgeleitet sind
unter anderem:

| Klasse | Inhalt |
|---|---|
| `DataFrame` | eine Tabelle (`pandas.DataFrame`) plus Metadaten je Spalte |
| `DataFrameGroup` | eine Sammlung von Tabellen, etwa die Proben einer Versuchsreihe |
| `Blob`, `FileReference`, `Image` | ein binärer Inhalt, direkt oder per URI referenziert, mit Prüfsumme |

Die Metadaten sind eine geordnete Sammlung von **Attributen**. Ein Attribut hat
die Felder:

| Feld | Bedeutung |
|---|---|
| `name` | Schlüssel des Attributs |
| `value` | Wert, umgewandelt in den deklarierten Typ |
| `dtype` | Datentyp, z. B. `float`, `timestamp`, `bytes`, `langstring` |
| `unit` | physikalische Einheit, z. B. `degC`, `kN`, `Hz` |
| `description` | Beschreibung in Worten |
| `label` | Anzeigename, z. B. für Achsenbeschriftungen |
| `symbol` | Formelzeichen der Größe, z. B. `T` |
| `ontology` | Klasse des Werts als CURIE oder IRI, z. B. `bfo:Quality` |
| `required` | Pflichtattribut im Sinne eines Schemas |

Eine Reihe von Attributen ist reserviert und beginnt mit `_sdata_`: Name,
lesbare Kennung, Klasse, Erzeugungszeitpunkt, Eltern- und Projektbezug,
Formatversion und weitere. Sie werden von der Bibliothek gepflegt; alle
übrigen Attribute sind frei. Ein `DataFrame` führt dieselbe Attributstruktur
noch einmal je Spalte (`column_metadata`), sodass jede Spalte eigene Einheit,
Bezeichnung, Formelzeichen und Klasse tragen kann.

Die Formatversion (`_sdata_format_version`) erlaubt es, ältere Dateien beim
Einlesen zu erkennen und zu migrieren, statt sie falsch zu interpretieren.

**Datentypen.** Die Typtabelle `sdata.dtypes` ist die einzige Quelle für die
Umwandlung von Attributwerten. Jeder Typ hat eine Python-Entsprechung und eine
Abbildung für JSON-LD:

| dtype | Python | JSON-LD |
|---|---|---|
| `str`, `int`, `float`, `bool` | `str`, `int`, `float`, `bool` | `xsd:string`, `xsd:integer`, `xsd:double`, `xsd:boolean` |
| `timestamp` | Zeitstempel mit Zeitzone | `xsd:dateTime` |
| `date`, `time`, `duration` | `datetime.date`, `datetime.time`, `datetime.timedelta` | `xsd:date`, `xsd:time`, `xsd:duration` |
| `decimal` | `decimal.Decimal` | `xsd:decimal` |
| `bytes` | `bytes` (in JSON als Base64) | `xsd:base64Binary` |
| `uri` | `str` | `xsd:anyURI` |
| `langstring` | Text mit Sprachkennung, z. B. `Hallo@de` | `rdf:langString` |
| `list`, `json` | Liste von Texten, `dict` | `xsd:string` |
| `complex`, `floatlist` | `complex`, Liste von Zahlen | `sdata:complex`, `sdata:floatlist` |

Eigene Typen lassen sich mit Umwandlungsfunktion und Abbildung über
`dtypes.register` ergänzen.

**Ablage.** Neben den Dateiformaten gibt es eine einheitliche Schnittstelle
aus Writern und Readern. Ein Writer nimmt ein `DataFrame` entgegen und legt
Daten und Metadaten so ab, wie es das Ziel verlangt: `ParquetWriter` schreibt
Parquet an eine beliebige `fsspec`-Adresse (auch als Verzeichnis mit einer
Datei je Tabelle), `StoreWriter` legt das ganze Objekt in einem
SQLite-Dokumentenspeicher ab und macht es über Name und Kennung auffindbar,
`SqlWriter` schreibt die Tabelle relational und die Metadaten in eine
Begleittabelle, und `GraphWriter` gibt die Metadaten als RDF aus. Die passenden
Reader lesen symmetrisch zurück. Maßgeblich für die Metadaten ist dabei stets
das `sdata`-Objekt selbst und nicht das Attribut `attrs` eines
`pandas.DataFrame`, das pandas bei vielen Operationen verwirft.

## 5 Einordnung in bestehende Standards

`sdata` steht nicht in Konkurrenz zu den folgenden Standards, sondern nutzt
sie. Die Tabelle zeigt, was jeweils übernommen wird.

| Standard | Zweck | Verwendung in sdata |
|---|---|---|
| Apache Parquet / Arrow | spaltenorientierte Tabellen | Datenformat; Metadaten in Schema- und Feld-Metadaten |
| HDF5 | hierarchische Binärdaten | ein Datensatz je Spalte; Metadaten als Attribute |
| Frictionless Data Package [3] | portables Datenpaket mit Deskriptor | `to_datapackage`: Standardfelder plus verlustfreier `sdata`-Block |
| CSV on the Web (CSVW) [4] | Beschreibung tabellarischer Daten | Spaltenliste im JSON-LD (`csvw:column`) |
| JSON-LD [5] | verknüpfte Daten in JSON | Ausgabe der Metadaten; eigener `@context` |
| QUDT [6] | Größen und Einheiten | Abbildung jeder Einheit auf `unit:…` |
| BFO [7] | Top-Level-Ontologie | Klassen für Objekte und Spaltenwerte |
| PROV, DCAT, Dublin Core | Herkunft, Kataloge | Erzeugungszeit, Ableitung, Zugehörigkeit |
| W3C DID [8] / Verifiable Credentials [9] | Identität und Signatur | `did:suuid:`-Kennungen; signierte Metadaten |

Gegenüber einem reinen Frictionless Data Package oder einer CSVW-Beschreibung
bietet `sdata` vor allem dreierlei: ein Objekt im Arbeitsspeicher, das Tabelle
und Metadaten gemeinsam hält, während man mit ihm rechnet; eine Einheitenlogik,
die über eine Textangabe hinausgeht; und dieselbe Metadatenstruktur in allen
Formaten. Umgekehrt beschränkt sich `sdata` auf das, was eine Python-Bibliothek
leisten kann: Es ist kein Repositorium, kein Katalogdienst und keine
Datenbank. Formate wie netCDF und Datenwürfel sind im Fahrplan vorgesehen, aber
nicht umgesetzt.

# Teil II — Ein durchgängiges Beispiel

Das Beispiel greift die Temperaturmessung aus dem Beitrag von 2020 auf. Es
setzt die Installation mit den Extras für Parquet, URI-Zugriff und Einheiten
voraus:

```
pip install "sdata[parquet,blob,units]"
```

Alle Ausgaben unten sind beim Erzeugen dieses Dokuments entstanden.

In [1]:
import logging, tempfile, pathlib
logging.disable(logging.WARNING)   # optionale Backends melden sich sonst beim Import
import pandas as pd
import sdata
print(f"sdata {sdata.__version__}, pandas {pd.__version__}")
out = pathlib.Path(tempfile.mkdtemp(prefix="sdata-paper-"))

sdata 1.6.2, pandas 3.1.1


## 6 Eine Tabelle anlegen

Ein `DataFrame` umhüllt eine gewöhnliche `pandas`-Tabelle und gibt ihr Namen
und Beschreibung. Die Tabelle selbst bleibt unter `.df` voll nutzbar.

In [2]:
from sdata.sclass.dataframe import DataFrame

df = pd.DataFrame({"time": [1.1, 2.1, 3.5],
                   "temperature": [2.4, 5.2, 2.2]})
messung = DataFrame(df=df, name="Temperaturmessung-001",
                    description="Messergebnis Temperatur")
messung.df

,time,temperature
index,,
0,1.1,2.4
1,2.1,5.2
2,3.5,2.2


## 7 Identität

Aus Klasse und Name entsteht die lesbare Kennung `sname`. Ein zweites,
unabhängig angelegtes Objekt mit demselben Namen erhält dieselbe Kennung —
die Messung lässt sich also an jedem Ort wiedererkennen, ohne dass eine
zentrale Stelle Nummern vergibt.

In [3]:
print("sname:", messung.sname)
again = DataFrame(df=df, name="Temperaturmessung-001")
print("deterministisch:", again.sname == messung.sname)

sname: DataFrame__temperaturmessung_001__e9a74fdd77205e81a07cd7f7eddc4cee
deterministisch: True


## 8 Spalten beschreiben

`set_column` setzt Einheit, Bezeichnung, Formelzeichen, Beschreibung und
Klasse einer Spalte. Nur die übergebenen Felder werden geändert.

In [4]:
messung.set_column("time", unit="s", label="Zeit", symbol="t",
                   description="Zeit seit Messbeginn")
messung.set_column("temperature", unit="degC", label="Temperatur", symbol="T",
                   description="Lufttemperatur", ontology="bfo:Quality")
messung.column_units

{'time': 's', 'temperature': 'degC'}

## 9 Metadaten der Messung

Metadaten, die die ganze Messung betreffen, kommen in `metadata`. Der
Messbeginn wird als Zeitstempel mit Zeitzone abgelegt, nicht als Text.

In [5]:
messung.metadata.add("sensor", "PT100", dtype="str", description="Messfühler")
messung.metadata.add("sampling_rate", 1.0, dtype="float", unit="Hz",
                     description="Abtastrate")
messung.metadata.add("start", "2026-11-06T09:00:00+01:00", dtype="timestamp",
                     description="Messbeginn")
messung.metadata.df.loc[["sensor", "sampling_rate", "start"],
                        ["value", "unit", "dtype", "description"]]

,value,unit,dtype,description
key,,,,
sensor,PT100,-,str,Messfühler
sampling_rate,1.0,Hz,float,Abtastrate
start,2026-11-06T09:00:00+01:00,-,timestamp,Messbeginn


Die reservierten Attribute pflegt die Bibliothek selbst:

In [6]:
[k for k in messung.metadata.df.index if k.startswith("_sdata_")]

['_sdata_class',
 '_sdata_ctime',
 '_sdata_format_version',
 '_sdata_name',
 '_sdata_parent_sname',
 '_sdata_project_sname',
 '_sdata_sname',
 '_sdata_suuid',
 '_sdata_topology_class',
 '_sdata_version']

Mit `strict=True` wird ein Wert, der nicht zum Typ passt, abgelehnt statt nur
gemeldet:

In [7]:
from sdata.dtypes import DtypeError
try:
    messung.metadata.add("offset", "klein", dtype="float", unit="K", strict=True)
except DtypeError as exc:
    print("abgelehnt:", exc)

abgelehnt: float: 'klein'


## 10 Einheiten

Das Einheitenmodul rechnet um, kennt die Dimension einer Einheit und die
Größe, für die sie steht. Der Dimensionsvektor hat die Achsenfolge
(L, M, T, Θ, A, I, N); das Volt ist also m²·kg·s⁻³·A⁻¹.

In [8]:
from sdata import units
print(units.convert(20.0, "degC", "K"))
print(units.convert(1.0, "kN", "N"))
print("dimension_of('V'):", units.dimension_of("V"))
print("quantity_of('ohm'):", units.quantity_of("ohm"))

293.15
1000.0
dimension_of('V'): (2, 1, -3, 0, 0, -1, 0)
quantity_of('ohm'): resistance


Eine ganze Tabelle lässt sich in ein Zielsystem von Einheiten überführen. Die
Spalteneinheiten werden dabei mitgeführt; das Zielsystem selbst wird in den
Metadaten vermerkt.

In [9]:
kelvin = messung.convert(units=["K", "s"])
print(kelvin.column_units)
kelvin.df

{'time': 's', 'temperature': 'K'}


,time,temperature
index,,
0,1.1,275.55
1,2.1,278.35
2,3.5,275.35


## 11 Serialisierung

Die Grundform jeder Serialisierung ist ein Python-Wörterbuch. Daraus
wiederhergestellt, ist das Objekt in Tabelle, Spalteneinheiten und Identität
gleich.

In [10]:
restored = DataFrame.from_dict(messung.to_dict())
print("Tabelle gleich:      ", restored.df.equals(messung.df))
print("Spalten-Units gleich:", restored.column_units == messung.column_units)
print("Identität gleich:    ", restored.sname == messung.sname)

Tabelle gleich:       True
Spalten-Units gleich: True
Identität gleich:     True


Dieselbe Messung als Parquet-Datei, als CSV mit JSON-LD-Begleitdatei und als
Frictionless Data Package:

In [11]:
messung.to_parquet(path=str(out))
messung.to_csv(path=str(out), sidecar=True)
messung.to_datapackage(path=str(out))
sorted(p.name for p in out.iterdir())

['DataFrame__temperaturmessung_001__e9a74fdd77205e81a07cd7f7eddc4cee.csv',
 'DataFrame__temperaturmessung_001__e9a74fdd77205e81a07cd7f7eddc4cee.meta.jsonld',
 'DataFrame__temperaturmessung_001__e9a74fdd77205e81a07cd7f7eddc4cee.spq',
 'DataFrame__temperaturmessung_001__e9a74fdd77205e81a07cd7f7eddc4cee.zip']

Beim Einlesen der Parquet-Datei kommen Name, Spalteneinheiten und alle
Metadaten zurück, auch der Zeitstempel. Er wird in UTC abgelegt: Der
Zeitpunkt bleibt gleich, die ursprüngliche Zeitzonenangabe nicht.

In [12]:
from_pq = DataFrame.from_parquet(str(out / f"{messung.sname}.spq"))
print(from_pq.name, from_pq.column_units)
print("sensor:", from_pq.metadata.get("sensor").value)
print("start: ", from_pq.metadata.get("start").value)

Temperaturmessung-001 {'time': 's', 'temperature': 'degC'}
sensor: PT100
start:  2026-11-06T08:00:00+00:00


Für viele Messungen eignet sich ein Store. `StoreWriter` legt das Objekt in
einer SQLite-Datei ab, `StoreReader` findet es über die Kennung wieder:

In [13]:
from sdata.iolib.writer import StoreWriter
from sdata.iolib.reader import StoreReader
writer = StoreWriter(str(out / "messungen.db"))
writer.write(messung)
writer.close()
reader = StoreReader(str(out / "messungen.db"))
aus_store = reader.read(messung.sname)
reader.close()
print(aus_store.name, aus_store.column_units)
print("start:", aus_store.metadata.get("start").value)

Temperaturmessung-001 {'time': 's', 'temperature': 'degC'}
start: 2026-11-06T08:00:00+00:00


Das Data Package enthält einen Standard-Deskriptor, den jedes
Frictionless-Werkzeug lesen kann. Einheit, Bezeichnung und Klasse der Spalten
stehen in den Feldern des Schemas:

In [14]:
import json, zipfile
with zipfile.ZipFile(out / f"{messung.sname}.zip") as zf:
    descriptor = json.loads(zf.read("datapackage.json"))
descriptor["resources"][0]["schema"]["fields"]

[{'name': 'time',
  'type': 'number',
  'title': 'Zeit',
  'description': 'Zeit seit Messbeginn',
  'unit': 's'},
 {'name': 'temperature',
  'type': 'number',
  'title': 'Temperatur',
  'description': 'Lufttemperatur',
  'unit': 'degC',
  'rdfType': 'bfo:Quality'}]

## 12 Maschinenlesbare Semantik

`to_jsonld` gibt die Metadaten als JSON-LD aus. Das Objekt erhält eine
`did:suuid:`-Kennung und die BFO-Klasse *independent continuant*; die
Temperaturspalte verweist auf die QUDT-Einheit `unit:DEG_C`.

In [15]:
doc = messung.to_jsonld()
print(json.dumps({k: doc[k] for k in ("@id", "@type", "name")}, indent=2))
print(json.dumps(doc["columns"][1], indent=2, ensure_ascii=False))

{
  "@id": "did:suuid:DataFrame__temperaturmessung_001__e9a74fdd77205e81a07cd7f7eddc4cee:sdata",
  "@type": [
    "sdata:DataFrame",
    "bfo:BFO_0000004"
  ],
  "name": "Temperaturmessung-001"
}
{
  "name": "temperature",
  "datatype": "xsd:double",
  "unitRef": "unit:DEG_C",
  "symbol": "degC",
  "rdfType": "bfo:Quality",
  "label": "Temperatur",
  "quantitySymbol": "T",
  "description": "Lufttemperatur"
}


## 13 Validierung gegen ein Schema

Ein `TableSchema` beschreibt, welche Spalten mit welchem Typ und welcher
Einheit erwartet werden. Verlangt das Schema die Temperatur in Kelvin, meldet
die Prüfung genau diese Spalte:

In [16]:
from sdata.schema import TableSchema, AttrSpec
schema = TableSchema("Temperaturmessung", [
    AttrSpec("time", dtype="float", unit="s", required=True),
    AttrSpec("temperature", dtype="float", unit="K", required=True),
])
print(messung.validate_table(schema))
print(kelvin.validate_table(schema))

ValidationReport(ok=False, missing=[], type_errors=[], unit_errors=['temperature'], enum_errors=[], extra=[], messages=[])
ValidationReport(ok=True, missing=[], type_errors=[], unit_errors=[], enum_errors=[], extra=[], messages=[])


## 14 Integrität

Eine Prüfsumme über den Tabelleninhalt zeigt, ob die Daten seit dem
Festhalten verändert wurden. Grundlage ist eine kanonische CSV-Form; dadurch
ist sie unabhängig vom Speicherformat reproduzierbar.

In [17]:
print("sha256:", messung.update_checksum()[:16], "…")
print("verify:", messung.verify())
messung.df.loc[1, "temperature"] = 5.3
print("nach Änderung:", messung.verify())
messung.df.loc[1, "temperature"] = 5.2
print("zurückgesetzt:", messung.verify())

sha256: 5377136fe3f9311b …
verify: True
nach Änderung: False
zurückgesetzt: True


## 15 Referenzierte Inhalte

Rohdaten, Bilder oder Protokolle müssen nicht kopiert werden. Ein `Blob`
verweist per URI auf den Inhalt (lokal, im Zip-Archiv oder, mit dem
passenden `fsspec`-Backend, etwa in S3) und hält dessen Prüfsumme fest.

In [18]:
from sdata.sclass.blob import Blob
ref = Blob(content_type="uri", value=str(out / f"{messung.sname}.csv"),
           filetype="csv", name="Rohdaten")
ref.update_checksum()
print(ref.exists(), ref.size, "Bytes, verify:", ref.verify())

True 41 Bytes, verify: True


Fehlt der Inhalt, sagt `exists()` das; ein Lesezugriff löst eine Ausnahme
aus, die zugleich `FileNotFoundError` und `ValueError` ist. Bestehender Code,
der `ValueError` abfängt, funktioniert dadurch weiter.

In [19]:
lost = Blob(content_type="uri", value=str(out / "fehlt.csv"), name="verloren")
print("exists:", lost.exists())
try:
    lost.content_bytes
except FileNotFoundError as exc:
    print(type(exc).__name__, isinstance(exc, ValueError))

exists: False
ContentNotFoundError True


## 16 Signierte Metadaten

Die Metadaten lassen sich als W3C Verifiable Credential signieren. Wer den
öffentlichen Schlüssel kennt, kann prüfen, dass sie unverändert vom
Aussteller stammen. Das Beispiel erzeugt dafür ein Ed25519-Schlüsselpaar.

In [20]:
from sdata.did import keys, pub_from_priv_jwk
priv = keys.gen_ed25519_jwk()
vc = messung.metadata.to_verifiable_credential("did:example:labor", priv)
print(vc[:48], "…")
checked = messung.metadata.from_verifiable_credential(vc, pub_from_priv_jwk(priv))
print("geprüft, sensor =", checked.get("sensor").value)

eyJhbGciOiJFZERTQSIsImtpZCI6ImRpZDpleGFtcGxlOmxh …
geprüft, sensor = PT100


# Grenzen und Ausblick

* **Abkündigung von `Data`.** Die Klasse aus dem Beitrag von 2020 gibt
  weiterhin eine Warnung aus und entfällt mit 2.0, ebenso einige experimentelle
  Module. Bestehende Dateien sollen dann über die Formatversion migriert
  werden.
* **Zeitzonen.** Zeitstempel werden in JSON-basierten Formaten in UTC
  geschrieben. Der Zeitpunkt ist eindeutig, die lokale Zeitzone der Messung geht
  dabei verloren; wer sie braucht, legt sie heute als eigenes Attribut ab.
* **Einheiten.** Logarithmische Größen wie Dezibel passen nicht in das lineare
  Modell aus Faktor und Versatz und sind bewusst ausgeklammert. Die
  Lichtstärke hat noch keine eigene Achse.
* **Formate.** netCDF und Datenwürfel sind vorgesehen, aber nicht umgesetzt.
  Eine relationale Ablage über SQL verliert die Spalteneinheiten, sofern sie
  nicht in einer Begleittabelle mitgeführt werden.
* **Reifegrad.** Das Paket ist als *Alpha* eingestuft; Schnittstellen können
  sich zwischen Nebenversionen noch ändern. Jede solche Änderung ist im
  Changelog als *Breaking* gekennzeichnet.

Die offenen Punkte sind im Repository unter `docs/rfc/open-points.md`
gesammelt.

# Fazit

Die Idee von 2020 — Daten, Metadaten und Beschreibung als ein Objekt — trägt
weiterhin. `sdata` 1.6 macht daraus ein Modell, in dem Einheiten berechenbar,
Typen geprüft und Identitäten reproduzierbar sind und das sich in etablierte
Formate und Vokabulare abbilden lässt, ohne dass die Metadaten unterwegs
verloren gehen. Die Temperaturmessung aus Teil II ist dieselbe wie vor sechs
Jahren; neu ist, dass sie heute als Parquet-Datei, Data Package oder signiertes
JSON-LD weitergegeben werden kann und beim Empfänger mit derselben Bedeutung
ankommt.

# Literatur und Verweise

1. I. Lepenies: *Das sdata-Format*. Jupyter-Notebook, 2020, sdata 0.8.4;
   im Repository unter `paper/2020/`.
2. M. D. Wilkinson et al.: The FAIR Guiding Principles for scientific data
   management and stewardship. *Scientific Data* 3, 160018 (2016).
   <https://doi.org/10.1038/sdata.2016.18>
3. Frictionless Data: Data Package. <https://datapackage.org/>
4. W3C: Metadata Vocabulary for Tabular Data (CSVW), 2015.
   <https://www.w3.org/TR/tabular-metadata/>
5. W3C: JSON-LD 1.1, 2020. <https://www.w3.org/TR/json-ld11/>
6. QUDT — Quantities, Units, Dimensions and Types. <https://qudt.org/>
7. Basic Formal Ontology (BFO). <https://basic-formal-ontology.org/>
8. W3C: Decentralized Identifiers (DIDs) v1.0, 2022.
   <https://www.w3.org/TR/did-core/>
9. W3C: Verifiable Credentials Data Model. <https://www.w3.org/TR/vc-data-model/>

**Software.** sdata, MIT-Lizenz: <https://github.com/lepy/sdata> ·
PyPI: <https://pypi.org/project/sdata/> · Dokumentation:
<https://lepy.github.io/sdata/> · DOI: 10.5281/zenodo.4311323